# YOLOv8 Model Retraining (3,300 Frames / 33 Batches)
### Combined Dataset: Batches 1–15 (1,500 frames) + Batches 16–52 (1,800 frames)

- **Total Dataset Size:** 3,300 annotated frames (2,640 train / 660 val) | 19,704 bounding boxes  
- **Classes:** `adult (0)`, `child (1)`  
- **Base Architecture:** `yolov8n.pt` (Nano)  
- **Original Configurations Preserved:** `imgsz=640`, `batch=8`, `epochs=100`, `workers=0`, `device=0` (RTX 2050 GPU), `close_mosaic=15`

In [ ]:
# Cell 1: Environment & GPU Device Check
import torch
import ultralytics
from ultralytics import YOLO

ultralytics.checks()

if torch.cuda.is_available():
    print(f'CUDA GPU Detected: {torch.cuda.get_device_name(0)}')
    device_id = 0
else:
    print('CUDA GPU not detected, falling back to CPU.')
    device_id = 'cpu'

## 🚀 Start Model Retraining (100 Epochs)

In [ ]:
# Cell 2: Retrain YOLOv8 Model with the Original Configurations
DATASET_YAML = r'C:/Users/Acer/Documents/CAPSTONE/web_dashboard_analytics/datasets/02_yolo_dataset/august11_33_batches/data.yaml'
PROJECT_DIR  = r'C:/Users/Acer/Documents/CAPSTONE/web_dashboard_analytics/datasets/03_training_runs'
RUN_NAME     = '04_august11_33_batches_3300frames'

model = YOLO('yolov8n.pt')

results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    workers=0,
    device=device_id,
    close_mosaic=15,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True
)

## 📊 Validate Retrained Model Checkpoint

In [ ]:
# Cell 3: Validate Retrained Checkpoint
best_weights = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=DATASET_YAML, split='val', device=device_id, imgsz=640)

print(f'Overall Precision:   {metrics.box.mp:.4f}')
print(f'Overall Recall:      {metrics.box.mr:.4f}')
print(f'Overall mAP@50:      {metrics.box.map50:.4f}')
print(f'Overall mAP@50-95:   {metrics.box.map:.4f}')

print('\nPer-Class Metrics:')
for idx, cname in enumerate(['adult (0)', 'child (1)']):
    print(f'  {cname}: P={metrics.box.p[idx]:.4f}, R={metrics.box.r[idx]:.4f}, mAP50={metrics.box.ap50[idx]:.4f}')